# Связка «Генератор SQL + Судья»: разбор без ML и математики

Этот ноутбук объясняет, как устроена наша система, **человеку без опыта в машинном обучении**. Формул не будет, код — с пояснением каждой строки, термины вводятся по ходу.

Система состоит из двух частей, работающих в паре:

1. **Генератор.** Превращает обычную просьбу («покажи договоры за месяц») в **SQL-запрос** — текст-команду для базы данных.
2. **Судья (аудитор).** Проверяет этот запрос на опасные ошибки до того, как он попадёт в базу.

> **Аналогия.** Генератор — автор, который пишет черновик. Судья — редактор-корректор, который вычитывает черновик и не пропускает опасные места, объясняя, что не так и как исправить.

**Зачем это нужно.** Если человек или нейросеть генерируют SQL, легко случайно написать запрос, который сотрёт все данные, утащит персональные данные или «положит» базу. Судья ловит такие случаи и заставляет переписать запрос.

**Как читать ноутбук:** нажимайте ▶️ на ячейках **по порядку, сверху вниз**. Устанавливать ничего не нужно — только стандартный Python.

## Три термина, которые пригодятся (простыми словами)

- **SQL** — язык запросов к базе данных. Запрос вроде `SELECT name FROM clients` означает «верни колонку name из таблицы clients». Базу можно представить как набор таблиц-Excel.
- **Нейросеть (LLM)** — программа, которая на огромном количестве текстов «научилась» продолжать текст и поэтому умеет писать код по описанию. Чтобы её **использовать**, математику знать не нужно: подаёшь текст-запрос — получаешь текст-ответ.
- **RAG** (retrieval-augmented generation) — приём «подложи нужную справку в запрос». Вместо того чтобы надеяться на память модели, мы сами находим релевантные примеры/правила и кладём их рядом с вопросом. Как дать студенту на экзамене нужную страницу учебника.

## Как части связаны

```
Просьба словами
   │
   ▼
Генератор ──► пишет SQL ──► Судья проверяет
   ▲                              │
   │ (урок: "так не делай")       ├─ безопасно → ГОТОВО
   └──── Reflector ◄──────────────┘ опасно → переписать
```

Двум частям помогают два вида RAG:
- **RAG #1 (примеры):** генератору — примеры **безопасных** запросов, судье — примеры **уязвимых**.
- **RAG #2 (знания):** судье — короткий **справочник**: чем опасно и как чинить, со ссылкой на стандарт (CWE).

## Панель 1 — Подключаем инструменты (`import`)

`import` — это «подключить готовый набор функций». Здесь всё встроено в Python, ставить ничего не надо.

In [ ]:
import re        # re — поиск в тексте по шаблону (например, "найти все email в тексте")
import json      # json — чтение/запись данных в текстовом формате (распространённый формат обмена)
import difflib   # difflib — оценка, насколько две строки похожи (число от 0 до 1)

print("Инструменты подключены")   # подтверждаем, что импорт прошёл успешно

## Панель 2 — Набор примеров (датасет)

**Датасет** — это просто размеченная таблица примеров, на которой мы и учим систему, и проверяем её качество. Каждая строка-«карточка»:
- `nl`   — просьба обычными словами (`nl` = natural language, естественный язык);
- `good` — безопасный вариант SQL;
- `bad`  — уязвимый вариант (есть только у проблемных примеров);
- `vuln` — тип уязвимости (класс) или `"safe"`, если пример безопасен.

> В ноутбуке 10 карточек для наглядности. В реальном проекте их **500**.

In [ ]:
DATASET = [                                                   # список карточек-примеров
    # ── Безопасные (safe): есть только корректный вариант ──
    {"nl": "топ-100 договоров по сумме кредита",               # просьба словами
     "good": "SELECT id, credit_amount FROM credit_contract WHERE status = 1 ORDER BY credit_amount DESC LIMIT 100",  # корректный SQL
     "vuln": "safe"},                                          # уязвимости нет
    {"nl": "сколько договоров в каждом подразделении",         # карточка 2
     "good": "SELECT org_id, COUNT(*) AS cnt FROM credit_contract GROUP BY org_id",  # агрегат — это нормально
     "vuln": "safe"},                                          # безопасно
    {"nl": "найти договор по номеру",                          # карточка 3
     "good": "SELECT id, credit_amount FROM credit_contract WHERE credit_contract_number = $1",  # $1 — безопасная подстановка значения
     "vuln": "safe"},                                          # безопасно

    # ── Проблемные: есть и bad (как нельзя), и good (как исправлено) ──
    {"nl": "показать договоры за месяц",                       # карточка 4
     "bad":  "SELECT * FROM credit_contract WHERE create_date > '2026-04-01' LIMIT 100",  # звёздочка * = все колонки
     "good": "SELECT id, credit_amount, create_date FROM credit_contract WHERE create_date > '2026-04-01' LIMIT 100",  # перечислили нужные
     "vuln": "SELECT_STAR"},                                   # тип: лишние/чувствительные колонки
    {"nl": "закрыть договор",                                  # карточка 5
     "bad":  "UPDATE credit_contract SET status = 0",          # без WHERE → изменит ВСЕ строки
     "good": "UPDATE credit_contract SET status = 0 WHERE id = $1",  # WHERE по id → только нужную
     "vuln": "DML_NO_WHERE"},                                  # тип: массовое изменение данных
    {"nl": "выгрузить все обороты",                            # карточка 6
     "bad":  "SELECT id, turnover_debit FROM count_turnover ORDER BY id DESC",  # без LIMIT → может вернуть миллионы строк
     "good": "SELECT id, turnover_debit FROM count_turnover ORDER BY id DESC LIMIT 1000",  # ограничили LIMIT
     "vuln": "NO_PAGINATION"},                                 # тип: нет пагинации
    {"nl": "найти договор по номеру из ввода пользователя",    # карточка 7
     "bad":  "SELECT id FROM credit_contract WHERE num = '\" + user_input + \"'",  # ввод пользователя вклеен в текст SQL
     "good": "SELECT id FROM credit_contract WHERE num = $1",  # безопасно — через параметр $1
     "vuln": "SQL_INJ_CLASSIC"},                              # тип: классическая SQL-инъекция
    {"nl": "поиск продукта по названию",                       # карточка 8
     "bad":  "SELECT id, name FROM dict_product WHERE name LIKE '%x%' UNION SELECT id, name FROM sys_object --",  # UNION к чужой таблице
     "good": "SELECT id, name FROM dict_product WHERE name LIKE $1",  # без UNION, через параметр
     "vuln": "SQL_INJ_UNION"},                                # тип: union-инъекция
    {"nl": "проверить договор по id",                          # карточка 9
     "bad":  "SELECT id FROM credit_contract WHERE id = 1 OR pg_sleep(3)",  # pg_sleep заставляет базу "ждать"
     "good": "SELECT id FROM credit_contract WHERE id = $1",   # безопасно — через параметр
     "vuln": "SQL_INJ_TIME"},                                 # тип: инъекция по времени отклика
    {"nl": "выгрузить контакты клиентов",                      # карточка 10
     "bad":  "SELECT full_name, passport, phone FROM sim_client",  # паспорт/телефон — персональные данные
     "good": "SELECT id, LEFT(phone,3) || '****' AS phone_masked FROM sim_client LIMIT 100",  # данные замаскированы
     "vuln": "DIRECT_SENSITIVE"},                             # тип: утечка персональных данных
]                                                             # конец списка

n_safe = sum(1 for r in DATASET if r["vuln"] == "safe")        # считаем безопасные карточки
n_bad  = sum(1 for r in DATASET if r["vuln"] != "safe")        # считаем проблемные карточки
print(f"В датасете {len(DATASET)} карточек: {n_safe} безопасных и {n_bad} с уязвимостями")  # сводка

## Панель 3 — Судья, часть 1: проверка по правилам

Первый слой судьи — **не нейросеть, а обычные правила** (поэтому он быстрый, предсказуемый и ничего не «выдумывает»). Это как **чек-лист таможенника**: фиксированный список того, что нельзя.

Каждое правило — отдельная небольшая функция: получает текст SQL и возвращает список **находок**. Находка — это словарь с полями: тип уязвимости, **оценка риска** (0–10), пояснение и ссылка на стандарт (CWE).

Дальше — **по одной панели на каждое правило** (всего 7).

> **Почему правилами, а не нейросетью?** Для явных опасных шаблонов («нет WHERE», «есть pg_sleep») правило надёжнее и дешевле: оно либо находит шаблон, либо нет, без вероятностей. Нейросеть подключим позже, для более тонких случаев и объяснений.

> 📚 **Почему гибрид (правила + LLM):** [ToxicSQL](https://arxiv.org/abs/2502.20527) — линтеры обходятся, один линтер не справляется; [QLPro](https://arxiv.org/abs/2506.23644) — LLM поверх статанализа (66% против 39% у CodeQL); [Valk Guard](https://github.com/ValkDB/valk-guard) и [Diesel Guard](https://github.com/ayarotsky/diesel-guard) — детерминированные правила-baseline по AST.


### Правило 1 — `SELECT *`

`SELECT *` запрашивает **все** колонки таблицы, включая ненужные и потенциально чувствительные. Лучше перечислять только то, что реально нужно.

> 📚 **Источник правила:** [Valk Guard](https://github.com/ValkDB/valk-guard) (19 PG-правил) + sqlfluff AM04. Чем грозит — [CWE-1295](https://cwe.mitre.org/data/definitions/1295.html).


In [ ]:
def rule_select_star(sql):                                     # на вход — текст SQL-запроса
    if "select *" in sql.lower():                              # ищем "select *" (.lower() — без учёта регистра)
        return [{"vuln": "SELECT_STAR", "risk": 5.0,          # нашли → возвращаем находку с риском 5
                 "msg": "SELECT * — возвращает все колонки, включая лишние/чувствительные",  # пояснение
                 "cwe": "CWE-1295"}]                          # ссылка на стандарт уязвимостей
    return []                                                 # не нашли → пустой список (нарушений нет)

### Правило 2 — `UPDATE`/`DELETE` без `WHERE`

`WHERE` ограничивает, какие строки затронуть. Без него изменение или удаление применится **ко всей таблице** — частая причина серьёзных аварий.

> 📚 **Источник правила:** [Valk Guard](https://github.com/ValkDB/valk-guard) (DML без WHERE — то, чего нет в sqlfluff). Класс — [CWE-1284](https://cwe.mitre.org/data/definitions/1284.html).


In [ ]:
def rule_dml_no_where(sql):                                    # на вход — текст SQL
    low = sql.lower().strip()                                  # к нижнему регистру и без пробелов по краям
    is_dml = low.startswith("update") or low.startswith("delete")  # это изменение/удаление данных?
    if is_dml and "where" not in low:                         # да, и нет слова where → опасно
        return [{"vuln": "DML_NO_WHERE", "risk": 9.0,         # риск 9 — высокий
                 "msg": "UPDATE/DELETE без WHERE затрагивает все строки таблицы",  # пояснение
                 "cwe": "CWE-1284"}]                          # ссылка
    return []                                                 # иначе — нарушений нет

### Правило 3 — `SELECT` без `LIMIT` и без `WHERE`

Чтение всей таблицы без ограничения может вернуть миллионы строк и перегрузить систему.
Тонкость: проверяем `count(` **со скобкой**, иначе имя таблицы `count_turnover` ошибочно примут за функцию `COUNT()`. Такие мелочи в правилах решают всё.

> 📚 **Источник правила:** sqlfluff AM09 / [Valk Guard](https://github.com/ValkDB/valk-guard); EXPLAIN-проверка тяжести — круг 3 (`research/03`). Класс — [CWE-770](https://cwe.mitre.org/data/definitions/770.html).


In [ ]:
def rule_no_limit(sql):                                        # на вход — текст SQL
    low = sql.lower()                                          # нижний регистр
    if not low.strip().startswith("select"):                   # правило только для чтения (SELECT)
        return []                                              # не SELECT → пропускаем
    if any(w in low for w in ["count(", "sum(", "avg(", "group by"]):  # это агрегат (итоговый расчёт)?
        return []                                              # агрегату LIMIT не нужен
    if "limit" in low or "where" in low:                      # есть ограничение или фильтр?
        return []                                              # значит выборка ограничена — ок
    return [{"vuln": "NO_PAGINATION", "risk": 4.0,            # иначе → находка с риском 4
             "msg": "SELECT без LIMIT и без WHERE может вернуть миллионы строк",  # пояснение
             "cwe": "CWE-770"}]                                # ссылка

### Правило 4 — классическая SQL-инъекция

Опасный шаблон: ввод пользователя **склеен** прямо в текст запроса. Тогда пользователь может дописать в запрос свою команду.

> **Аналогия.** Это как разрешить посетителю самому дописывать пункты в служебную инструкцию: он впишет что угодно. Правильно — принимать значение в отдельное «окошко» (параметр `$1`), а не вклеивать в текст.

> 📚 **Источник:** payloads [PortSwigger](https://portswigger.net/web-security/sql-injection/cheat-sheet) и [sqlmap](https://github.com/sqlmapproject/sqlmap); [P2SQL (инъекции через LLM-агентов)](https://arxiv.org/abs/2308.01990). Класс — [CWE-89](https://cwe.mitre.org/data/definitions/89.html), [CAPEC-66](https://capec.mitre.org/data/definitions/66.html).


In [ ]:
def rule_sqli_classic(sql):                                    # на вход — текст SQL
    pattern = r"['\"]\s*\+\s*\w+\s*\+\s*['\"]"                # шаблон склейки: кавычка + слово + кавычка через "+"
    if re.search(pattern, sql):                                # такой шаблон встречается в тексте?
        return [{"vuln": "SQL_INJ_CLASSIC", "risk": 10.0,    # риск 10 — максимальный
                 "msg": "Ввод пользователя склеен в текст SQL — инъекция",  # пояснение
                 "cwe": "CWE-89"}]                            # ссылка
    return []                                                 # иначе — нарушений нет

### Правило 5 — `UNION`-инъекция

`UNION` присоединяет к ответу ещё одну выборку. Это опасно, если присоединяют **чужую/системную** таблицу или обрезают конец запроса через `--`, чтобы достать данные, к которым нет доступа.

> 📚 **Источник:** sqlmap union-payloads (адаптированы под схему). Класс — CWE-89, [CAPEC-66](https://capec.mitre.org/data/definitions/66.html).


In [ ]:
def rule_union(sql):                                           # на вход — текст SQL
    low = sql.lower()                                          # нижний регистр
    has_union = "union" in low                                 # присутствует UNION?
    suspicious = ("sys_object" in low or "--" in low or "select null" in low)  # признаки атаки
    if has_union and suspicious:                               # UNION + подозрительный признак → опасно
        return [{"vuln": "SQL_INJ_UNION", "risk": 8.0,       # риск 8
                 "msg": "UNION к чужой/системной таблице — попытка достать недоступные данные",  # пояснение
                 "cwe": "CWE-89"}]                            # ссылка
    return []                                                 # иначе — нарушений нет

### Правило 6 — `pg_sleep` (инъекция по времени)

`pg_sleep(3)` заставляет базу «подождать» 3 секунды. Атакующий по наличию/отсутствию задержки делает вывод «да/нет» и так по крупицам вытягивает данные, даже не видя их напрямую.

> 📚 **Источник:** time-based blind payloads (sqlmap). Класс — CWE-89, [CAPEC-7 (Blind SQLi)](https://capec.mitre.org/data/definitions/7.html).


In [ ]:
def rule_pg_sleep(sql):                                        # на вход — текст SQL
    if "pg_sleep" in sql.lower():                              # встречается ли pg_sleep?
        return [{"vuln": "SQL_INJ_TIME", "risk": 9.0,        # риск 9
                 "msg": "pg_sleep — инъекция по времени отклика (blind injection)",  # пояснение
                 "cwe": "CWE-89"}]                            # ссылка
    return []                                                 # иначе — нарушений нет

### Правило 7 — чувствительные колонки

Некоторые поля — персональные/секретные данные (паспорт, телефон, CVV). Их нельзя возвращать «как есть». Если значение **замаскировано** (есть алиас `AS ..._masked`) — это допустимо.

> 📚 **Источник:** [Microsoft Presidio](https://github.com/microsoft/presidio) (детект PII) + RU-валидаторы (СНИЛС/ИНН); [Schema-inference attack](https://arxiv.org/abs/2406.14545) — почему нельзя светить схему/данные. Класс — [CWE-200](https://cwe.mitre.org/data/definitions/200.html).


In [ ]:
SENSITIVE = ["passport", "snils", "cvv", "card_number",        # перечень чувствительных имён колонок
             "password", "phone", "email", "full_name"]       # паспорт, СНИЛС, телефон, ФИО и т.п.

def rule_sensitive(sql):                                        # на вход — текст SQL
    low = sql.lower()                                          # нижний регистр
    part = low.split(" from ")[0]                              # берём часть ДО " from " — это перечень выбираемых колонок
    if "as " in part:                                         # если есть алиас "AS ..." → значение замаскировано
        return []                                              # маскированное — допустимо, пропускаем
    for col in SENSITIVE:                                      # перебираем каждое чувствительное имя
        if col in part:                                        # такая колонка среди выбираемых?
            return [{"vuln": "DIRECT_SENSITIVE", "risk": 7.0, # риск 7
                     "msg": f"Прямая выдача чувствительных данных ({col})",  # пояснение с именем поля
                     "cwe": "CWE-200"}]                       # ссылка
    return []                                                 # чувствительного нет — нарушений нет

### Собираем правила в один список

Чтобы потом прогонять запрос сразу через все правила, складываем функции в список.

In [ ]:
RULES = [                                                      # перечень всех правил-проверок
    rule_select_star,                                         # 1) SELECT *
    rule_dml_no_where,                                        # 2) UPDATE/DELETE без WHERE
    rule_no_limit,                                            # 3) SELECT без LIMIT/WHERE
    rule_sqli_classic,                                        # 4) склейка ввода (инъекция)
    rule_union,                                               # 5) UNION-инъекция
    rule_pg_sleep,                                            # 6) pg_sleep
    rule_sensitive,                                           # 7) чувствительные колонки
]                                                            # конец списка
print(f"Подключено правил: {len(RULES)}")                     # сколько правил собрали

## Панель 4 — Вердикт судьи и порог риска

Логика проста: прогнать запрос через все правила → взять **максимальный** риск среди находок → сравнить с **порогом 4.0**.

- риск **меньше 4** → запрос **одобрен**;
- риск **4 и выше** → **отклонён**.

Берём максимум (а не сумму), чтобы одна серьёзная проблема не «размывалась» мелкими. Порог 4.0 — настройка из техзадания.

> 📚 **Источник:** baseline-контракт `RISK_THRESHOLD=4.0` (ТЗ, ADR-0004); агрегация голосов/риска — [QLPro](https://arxiv.org/abs/2506.23644) (triple-voting), [Надёжность LLM-судьи](https://arxiv.org/abs/2412.12509) (minority-veto).


In [ ]:
THRESHOLD = 4.0                                                # порог одобрения (из ТЗ)

def judge(sql):                                                # на вход — SQL, на выход — вердикт
    findings = []                                             # сюда соберём все находки
    for rule in RULES:                                        # прогоняем по очереди каждое правило
        findings += rule(sql)                                 # добавляем его находки к общему списку
    risk = max([f["risk"] for f in findings], default=0.0)    # максимальный риск (если находок нет → 0)
    approved = risk < THRESHOLD                               # ниже порога → одобрено
    return {"approved": approved, "risk": risk, "findings": findings}  # возвращаем вердикт

### Быстрая проверка

In [ ]:
print(judge("UPDATE credit_contract SET status = 0"))         # уязвимый → ожидаем отклонение (risk 9)
print(judge("SELECT id FROM credit_contract WHERE id = $1"))  # безопасный → ожидаем одобрение (risk 0)

## Панель 5 — Проверяем судью на всём датасете

Прогоняем все уязвимые запросы (судья должен их ловить) и все безопасные (не должен срабатывать впустую).

In [ ]:
print("Уязвимые запросы (должны быть отклонены):")           # заголовок
for r in DATASET:                                             # перебираем карточки
    if "bad" in r:                                            # только те, где есть уязвимый вариант
        v = judge(r["bad"])                                   # судим уязвимый запрос
        mark = "отклонён ✓" if not v["approved"] else "ПРОПУЩЕН ✗"  # ожидаем "отклонён"
        print(f"  {mark:14} риск={v['risk']:.0f}  [{r['vuln']}]")  # печатаем итог

print("\nБезопасные запросы (должны быть одобрены):")          # заголовок
for r in DATASET:                                             # перебираем карточки
    v = judge(r["good"])                                      # судим безопасный вариант
    mark = "одобрен ✓" if v["approved"] else "ложно отклонён ✗"  # ожидаем "одобрен"
    print(f"  {mark:18} риск={v['risk']:.0f}")                # печатаем итог

## Панель 6 — RAG #1: подбор похожих примеров (асимметрично)

Напомним: **RAG** — это «подложить нужную справку». Здесь справка — это **примеры из датасета**. Две раздельные стопки:
- **безопасные** примеры → отдаём только **генератору** (чтобы писал по образцу);
- **уязвимые** примеры → отдаём только **судье** (чтобы узнавал известные атаки).

> **Почему раздельно («асимметрично»).** Логика как в обучении: исполнителю показывают эталоны, а контролёру — каталог типовых нарушений. Смешивать нельзя: генератор не должен «насмотреться» на уязвимые примеры и начать их повторять.

«Похожесть» здесь считаем самым простым способом — сравнением строк (`difflib`), **без векторов и математики**. В реальном проекте это умные эмбеддинги, но идея та же: найти наиболее близкие по смыслу примеры.

> 📚 **Источники (ядро идеи, асимметрия):** [Vul-RAG](https://arxiv.org/abs/2406.11147) — негативные примеры судье (голая LLM различает vuln/patched лишь 0.06–0.14!); [DAIL-SQL](https://arxiv.org/abs/2308.15363) — позитивные few-shot генератору + self-consistency; [Сложность выбора few-shot для детекции](https://arxiv.org/abs/2510.27675) — брать примеры, на которых модель ошибается. Сама **асимметрия** (pos→генератору, neg→судье) — наша новизна, ADR-0012.


### Шаг 1 — разделить датасет на две стопки

In [ ]:
POSITIVES = [r for r in DATASET]                              # стопка для генератора (у всех есть good)
NEGATIVES = [r for r in DATASET if "bad" in r]               # стопка для судьи (только с bad)
print("безопасных:", len(POSITIVES), " уязвимых:", len(NEGATIVES))  # размеры стопок

### Шаг 2 — функция «насколько похожи две фразы»

Возвращает число от 0 до 1: 0 — совсем разные, 1 — идентичные. Никакой математики — встроенное сравнение строк.

In [ ]:
def similar(a, b):                                            # на вход — две фразы
    a, b = a.lower(), b.lower()                               # обе к нижнему регистру (сравниваем честно)
    return difflib.SequenceMatcher(None, a, b).ratio()       # доля совпадения: число от 0 до 1

### Шаг 3 — подобрать БЕЗОПАСНЫЕ примеры (для генератора)

In [ ]:
def retrieve_positive(task, k=2):                             # task — просьба словами, k — сколько вернуть
    ranked = sorted(POSITIVES,                                # сортируем стопку безопасных...
                    key=lambda r: similar(task, r["nl"]),    # ...по похожести их nl на просьбу
                    reverse=True)                            # самые похожие — в начало
    return [r["good"] for r in ranked[:k]]                   # берём первые k и возвращаем их good-SQL

### Шаг 4 — подобрать УЯЗВИМЫЕ примеры (для судьи)

In [ ]:
def retrieve_negative(task, k=2):                             # task — просьба, k — сколько вернуть
    ranked = sorted(NEGATIVES,                                # сортируем стопку уязвимых...
                    key=lambda r: similar(task, r["nl"]),    # ...по похожести
                    reverse=True)                            # самые похожие — в начало
    return [(r["vuln"], r["bad"]) for r in ranked[:k]]       # возвращаем пары (тип, уязвимый SQL)

### Проверим обе стопки

In [ ]:
print("Генератору для 'договоры по сумме':")                  # заголовок
for s in retrieve_positive("договоры по сумме"):             # подбираем безопасные примеры
    print("  +", s[:60])                                     # печатаем (первые 60 символов)
print("Судье для 'закрыть договор':")                         # заголовок
for v, s in retrieve_negative("закрыть договор"):            # подбираем уязвимые примеры
    print(f"  - [{v}] {s[:50]}")                             # печатаем тип + SQL

## Панель 7 — RAG #2: справочник знаний (CWE и рекомендации)

Когда правило что-то нашло, судья обращается к **справочнику**: чем это опасно, как исправить и ссылка на стандарт (CWE — общепринятая нумерация типов уязвимостей).

> **Аналогия.** Это как медицинский справочник: по симптому — диагноз, причина, рекомендация и ссылка на источник. Важно, что вывод **обоснован ссылкой**, а не «придуман» — это и есть прозрачность отчёта.

> 📚 **Источники:** [ProveRAG](https://arxiv.org/abs/2410.17406) — provenance/ссылки на CWE-NVD против галлюцинаций; [RAVEN](https://arxiv.org/abs/2604.17948) — курируемая база CWE + роли Explorer/Analyst/Reporter; [AgentAuditor](https://arxiv.org/abs/2506.00641) — память опыта + multi-stage RAG; [RAG+Self-Ranking для SQLi-детекции](https://arxiv.org/abs/2411.18216) (+66 п.п. F2). База знаний — ADR-0005.


### Шаг 1 — сам справочник

In [ ]:
KNOWLEDGE = {                                                  # ключ = тип уязвимости, значение = (чем опасно, как чинить, CWE)
    "SELECT_STAR":      ("возвращает лишние/чувствительные колонки", "перечислять нужные колонки явно", "CWE-1295"),
    "DML_NO_WHERE":     ("изменяет все строки сразу",               "добавить WHERE по id",            "CWE-1284"),
    "NO_PAGINATION":    ("может вернуть миллионы строк",            "добавить LIMIT",                  "CWE-770"),
    "SQL_INJ_CLASSIC":  ("ввод склеен в запрос",                   "использовать параметр $1",        "CWE-89"),
    "SQL_INJ_UNION":    ("доступ к чужим таблицам",                "параметризовать, запретить UNION","CWE-89"),
    "SQL_INJ_TIME":     ("инъекция по времени (pg_sleep)",         "параметр $1 + тайм-аут",          "CWE-89"),
    "DIRECT_SENSITIVE": ("утечка персональных данных",             "маскировать или агрегировать",    "CWE-200"),
}                                                            # конец справочника

### Шаг 2 — функция «объясни уязвимость»

In [ ]:
def explain(vuln):                                            # на вход — тип уязвимости
    why, fix, cwe = KNOWLEDGE.get(vuln,                       # достаём тройку из справочника...
                                  ("неизвестно", "проверить вручную", "-"))  # ...или заглушку, если типа нет
    return f"чем опасно: {why} | как чинить: {fix} | источник: {cwe}"  # собираем читаемую строку

print(explain("DML_NO_WHERE"))                                # пример вывода справочника

## Панель 8 — Генератор (автор SQL)

В реальном проекте здесь работает нейросеть (Qwen-Coder). Чтобы ноутбук запускался без интернета и ключей, сделаем **упрощённую имитацию**, которая ведёт себя показательно:
- пока **не было замечаний** — выдаёт намеренно небезопасный вариант;
- получив **урок** от Reflector — исправляется.

Так на простом коде видно поведение настоящей связки. Ниже — отдельная панель с тем, как подключить реальную нейросеть.

> 📚 **Источники:** [DAIL-SQL](https://arxiv.org/abs/2308.15363) — code-representation промпт; [MAG-SQL](https://arxiv.org/abs/2408.07930) — декомпозиция + schema linking (+14.7 п.п.); [CHASE-SQL](https://arxiv.org/abs/2410.01943) — несколько кандидатов + выбор лучшего. Модель ≤30B (Qwen-Coder) — ADR-0008.


In [ ]:
def generator(task, lessons, positives):                      # task — просьба, lessons — уроки, positives — примеры
    # Настоящая нейросеть собрала бы из task + positives + lessons текст-промпт и вернула SQL.
    if not lessons:                                          # если замечаний ещё не было...
        return "SELECT * FROM credit_contract"               # ...имитируем небезопасный вариант (звёздочка, без where/limit)
    return "SELECT id, status FROM credit_contract WHERE status = 1 LIMIT 100"  # с уроком → корректный вариант

print(generator("покажи договоры", [], []))                   # без уроков → небезопасно
print(generator("покажи договоры", ["не используй *"], []))   # с уроком → безопасно

### (Необязательно) Подключение настоящей нейросети

Если есть API-ключ (например, OpenRouter), генератор можно заменить на реальную модель. Раскомментируйте код и вставьте ключ. Математика для этого не нужна — это обычный вызов сервиса по сети.

In [ ]:
# !pip install openai                                          # установить клиент (раскомментировать при запуске)
# from openai import OpenAI                                     # импортировать клиент
# client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key="ВАШ_КЛЮЧ")  # подключиться к сервису
# def generator(task, lessons, positives):                      # заменяем имитацию на реальную модель
#     prompt = "Примеры:\n" + "\n".join(positives) + "\nУроки:\n" + "\n".join(lessons) + "\nЗадача: " + task  # собираем текст-запрос
#     r = client.chat.completions.create(                       # отправляем запрос модели
#         model="qwen/qwen-2.5-coder-32b-instruct",            # модель до 30B (требование кейса)
#         messages=[{"role": "user", "content": prompt}])      # передаём промпт
#     return r.choices[0].message.content                      # возвращаем сгенерированный SQL
print("Справочная ячейка: подключать реальную модель не обязательно для понимания.")  # подсказка

## Панель 9 — Reflector (превращает находки в уроки)

Если судья что-то нашёл, Reflector формулирует короткие **уроки** и передаёт их генератору на следующий круг.

> **Аналогия.** Корректор оставляет автору пометку на полях: «здесь не вклеивай ввод, используй параметр». В следующий раз автор её учитывает. Это «память на замечаниях» — без переобучения самой модели.

> 📚 **Источники:** [Reflexion](https://arxiv.org/abs/2303.11366) — «вербальный RL», память об ошибках (+11% code); [Self-Refine](https://arxiv.org/abs/2303.17651) — одна модель генерит→критикует→правит (+20%); [PREFER](https://arxiv.org/abs/2308.12033) — feedback-reflect-refine.


In [ ]:
def reflect(findings, lessons):                               # findings — что нашёл судья, lessons — уже накопленные уроки
    for f in findings:                                       # по каждой находке
        _, fix, _ = KNOWLEDGE.get(f["vuln"], ("", "исправить", ""))  # берём из справочника рекомендацию
        lesson = f"[{f['vuln']}] {fix}"                       # формулируем урок
        if lesson not in lessons:                            # чтобы не дублировать одно и то же
            lessons.append(lesson)                           # добавляем новый урок
    return lessons                                           # возвращаем обновлённый список уроков

print(reflect([{"vuln": "SELECT_STAR"}], []))                 # пример: из находки получили урок

## Панель 10 — Полный цикл

1. генератор пишет SQL (видит безопасные примеры и накопленные уроки);
2. судья проверяет;
3. одобрено → готово; отклонено → Reflector формулирует урок → снова к шагу 1; не больше 3 кругов, чтобы не зациклиться.

> **Аналогия.** Черновик → правки корректора → переработанная версия → снова на проверку.

> 📚 **Источники:** [MAC-SQL Refiner](https://arxiv.org/abs/2312.11242) (execution-feedback → у нас security-feedback); [MSC-SQL](https://arxiv.org/abs/2410.12916), [SCoT2S](https://www.sciencedirect.com/science/article/abs/pii/S0885230825000907) — multi-sample и Generate→Detect→Correct.


In [ ]:
def run_pipeline(task, max_iters=3):                          # task — просьба, max_iters — лимит кругов
    lessons = []                                             # накопленные уроки (в начале пусто)
    log = []                                                 # журнал: что произошло на каждом круге
    sql, verdict = "", None                                  # последний SQL и последний вердикт

    for it in range(1, max_iters + 1):                       # круги с 1 по max_iters
        positives = retrieve_positive(task)                 # RAG #1: подбираем примеры генератору
        sql = generator(task, lessons, positives)           # 1) генератор пишет SQL
        verdict = judge(sql)                                # 2) судья проверяет
        log.append((it, sql, verdict, list(lessons)))       # записываем круг в журнал
        if verdict["approved"]:                             # 3) если одобрено...
            break                                           # ...выходим из цикла
        lessons = reflect(verdict["findings"], lessons)     # иначе формулируем уроки и идём дальше

    return {"final_sql": sql, "approved": verdict["approved"], "log": log}  # итог работы

### Функция отчёта (прозрачный журнал аудита)

Этот журнал — то, что увидит человек: что было сделано, какие риски найдены, со ссылками и рекомендациями.

In [ ]:
def show(result):                                             # на вход — результат run_pipeline
    print("=" * 60)                                          # разделитель
    for it, sql, v, lessons in result["log"]:               # по каждому кругу из журнала
        print(f"\n— Круг {it} —")                            # номер круга
        print("  SQL:", sql)                                # какой SQL сгенерирован
        status = "ОДОБРЕНО" if v["approved"] else "ОТКЛОНЕНО"  # вердикт словом
        print(f"  Риск: {v['risk']:.0f}  →  {status}")       # риск и вердикт
        for f in v["findings"]:                             # по каждой находке
            print(f"    [!] {f['vuln']}: {f['msg']}")        # тип уязвимости и пояснение
            print(f"        {explain(f['vuln'])}")           # обоснование из справочника (RAG #2)
        if lessons:                                          # если уже есть уроки
            print("  Уроки:", lessons)                       # показываем их
    print("\nИТОГ:", result["final_sql"], "| одобрено:", result["approved"])  # финальный результат
    print("=" * 60)                                          # разделитель

## Панель 11 — Запуск

Видно, как за пару кругов небезопасный запрос превращается в корректный.

In [ ]:
res = run_pipeline("покажи кредитные договоры за месяц")      # запускаем весь цикл
show(res)                                                     # печатаем журнал

## Панель 12 — Как измерить качество судьи

Две понятные метрики (это **просто проценты**, без формул):
- **Recall (полнота)** — какую долю уязвимых запросов судья поймал. Цель — 100%.
- **Ложные срабатывания** — какую долю безопасных запросов он отклонил зря. Цель — 0%.

> **Аналогия.** Хороший контролёр находит все настоящие нарушения и при этом не штрафует невиновных.

> 📚 **Источники метрик:** [ETM (Enhanced Tree Matching)](https://arxiv.org/abs/2407.07313) — честная оценка SQL (FP/FN 0.3%/2.7%); [SecureSQL](https://aclanthology.org/2024.findings-emnlp.346/) — метрики по типам атак; Execution Accuracy — круг 2 (`research/02`).


### Шаг 1 — отобрать уязвимые и безопасные

In [ ]:
bad  = [r for r in DATASET if "bad" in r]                     # все примеры с уязвимым вариантом
good = [r for r in DATASET]                                   # у всех есть безопасный вариант
print("уязвимых:", len(bad), " безопасных:", len(good))       # размеры групп

### Шаг 2 — посчитать Recall

In [ ]:
caught = sum(1 for r in bad if not judge(r["bad"])["approved"])  # уязвимый отклонён = пойман
recall = caught / len(bad) * 100                              # доля пойманных, в процентах
print(f"Recall: {caught}/{len(bad)} = {recall:.0f}%")          # выводим

### Шаг 3 — посчитать ложные срабатывания

In [ ]:
false_alarm = sum(1 for r in good if not judge(r["good"])["approved"])  # безопасный отклонён = ошибка
fp = false_alarm / len(good) * 100                           # доля ложных срабатываний, в процентах
print(f"Ложные срабатывания: {false_alarm}/{len(good)} = {fp:.0f}%")  # выводим
print("\nЦель: Recall 100%, ложных срабатываний 0%.")        # ориентир

## Панель 13 — Связь с реальным проектом и что дальше

Это упрощённая, но **идейно точная** версия. В рабочем проекте (`src/case3/`) то же самое, но «по-взрослому»:

| В этом ноутбуке | В проекте `src/case3/` |
|---|---|
| 7 правил в отдельных панелях | `nodes/auditor.py` — правила R001–R013 (recall 100%) |
| `judge()` с порогом | `HybridAuditor`: правила + слой нейросети |
| `retrieve_positive/negative` | `retrieval/fewshot.py` (асимметричный подбор примеров) |
| `KNOWLEDGE` + `explain()` | `audit/knowledge.py` (CWE/CAPEC/OWASP) |
| имитация генератора | `nodes/generator.py` + реальная **Qwen-Coder** |
| `run_pipeline()` | `pipeline.py` (в планах — на **LangGraph**) |
| 10 примеров | `data/dataset_v1.jsonl` — **500** |

**Что можно сделать дальше:**
1. подключить настоящую нейросеть (панель 8);
2. заменить простое сравнение строк на эмбеддинги (e5 + FAISS) — точнее ищет похожие примеры;
3. прогнать метрики на всех 500 примерах.

Итог: вы разобрались, как устроена связка «генератор + судья», какие у неё части и зачем каждая нужна — без машинного обучения и математики.

## Панель 14 — На каких исследованиях построено решение

Решение собрано как «лоскутное одеяло» из проверенных работ: из каждой взята конкретная фишка. Полный реестр (с тем, что именно взяли) — ниже. Карточки-разборы лежат в репозитории: `research/materials/` и `research/materials_round6/`.

### Архитектура цикла «генератор ↔ судья»
| Работа | Что взяли | Ссылка |
|---|---|---|
| MAC-SQL (COLING 2025) | каркас Selector→Decomposer→Refiner; заменили execution-feedback на security-feedback | https://arxiv.org/abs/2312.11242 |
| LangGraph Reflection Agents | паттерн узлов generator↔critic, лимит итераций | https://blog.langchain.com/reflection-agents/ |
| R³ Review-Rebuttal-Revision | цикл «предложил→раскритиковали→переписал» до консенсуса | https://aclanthology.org/2025.trl-1.4/ |
| Reflexion (NeurIPS 2023) | память об ошибках без дообучения (наш Reflector) | https://arxiv.org/abs/2303.11366 |
| Self-Refine (2023) | одна модель генерит→критикует→правит | https://arxiv.org/abs/2303.17651 |
| PREFER | feedback-reflect-refine: уроки → новые промпты | https://arxiv.org/abs/2308.12033 |

### Генератор (NL → SQL)
| Работа | Что взяли | Ссылка |
|---|---|---|
| DAIL-SQL | code-representation промпт + self-consistency | https://arxiv.org/abs/2308.15363 |
| MAG-SQL | декомпозиция Targets/Conditions + schema linking | https://arxiv.org/abs/2408.07930 |
| CHASE-SQL (ICLR 2025) | генерация нескольких кандидатов + выбор лучшего | https://arxiv.org/abs/2410.01943 |
| MSC-SQL | multi-sample генерация + критик-оценка | https://arxiv.org/abs/2410.12916 |
| SCoT2S | трёхэтапка Generate→Detect→Correct | https://www.sciencedirect.com/science/article/abs/pii/S0885230825000907 |
| RetrySQL | данные «ошибка→исправление» (идея промпта) | https://arxiv.org/abs/2507.02529 |
| SQLCritic | пощёлочная (clause-wise) критика | https://arxiv.org/abs/2503.07996 |
| ErrorLLM | error-токены вместо текстовых ошибок | https://arxiv.org/abs/2603.03742 |

### Судья: детерминированный слой (правила)
| Работа | Что взяли | Ссылка |
|---|---|---|
| Valk Guard | 19 PG-правил без подключения к БД (SELECT*, DML без WHERE, no-LIMIT) | https://github.com/ValkDB/valk-guard |
| Diesel Guard | libpg_query AST-парсинг идентично PostgreSQL | https://github.com/ayarotsky/diesel-guard |
| pglast (круг 3) | нативный PG-парсер для перевода правил с regex на AST | https://github.com/lelit/pglast |
| ToxicSQL | доказательство: один линтер обходится (нужен гибрид) | https://arxiv.org/abs/2502.20527 |

### Судья: LLM-слой + RAG знаний
| Работа | Что взяли | Ссылка |
|---|---|---|
| QLPro | triple-voting LLM-судьи поверх статанализа | https://arxiv.org/abs/2506.23644 |
| Vul-RAG | знаниевый RAG по парам «уязвимость+фикс» (негативы судье) | https://arxiv.org/abs/2406.11147 |
| ProveRAG | provenance: ссылки на CWE/NVD против галлюцинаций | https://arxiv.org/abs/2410.17406 |
| RAVEN | курируемая база CWE + роли Explorer/Analyst/Reporter | https://arxiv.org/abs/2604.17948 |
| AgentAuditor | память опыта + multi-stage context-aware RAG | https://arxiv.org/abs/2506.00641 |
| RobustJudge | устойчивость судьи к prompt-инъекциям (15 атак / 7 защит) | https://arxiv.org/abs/2506.09443 |
| RAG+Self-Ranking (SQLi) | RAG поднимает F2-детекции SQLi до +66 п.п. | https://arxiv.org/abs/2411.18216 |
| MulVul | RAG + мультиагент: Router→Detector по классам | https://arxiv.org/abs/2601.18847 |

### Безопасность (что судья обязан ловить)
| Работа | Что взяли | Ссылка |
|---|---|---|
| P2SQL | инъекции через LLM-агентов + 4 защиты | https://arxiv.org/abs/2308.01990 |
| Schema-inference attack | маскировать схему/данные в выводе | https://arxiv.org/abs/2406.14545 |
| GSQLi (GAN, WAF-bypass) | тест замаскированных SQLi | IEEE ICCAI 2025 |
| PortSwigger / sqlmap | реальные payload-паттерны (адаптированы под схему) | https://portswigger.net/web-security/sql-injection/cheat-sheet |
| Presidio + RU-валидаторы | детект PII (паспорт/СНИЛС/ИНН), Luhn/mod-101 | https://github.com/microsoft/presidio |
| PostgreSQL CVE-2025-1094/8714/8715 | PG-специфика (escape, pg_dump, psql meta) | https://nvd.nist.gov/vuln/detail/CVE-2025-1094 |

### Ансамбли / голосование / boosting (трек ВКР, ADR-0012)
| Работа | Что взяли | Ссылка |
|---|---|---|
| Self-MoA (Rethinking MoA) | под ≤30B лучше ансамбль одной сильной модели | https://arxiv.org/abs/2502.00674 |
| PoLL (жюри судей) | панель мелких моделей > одного крупного, ×7 дешевле | https://arxiv.org/abs/2404.18796 |
| More Agents Is All You Need | sampling-and-voting масштабируется с числом агентов | https://arxiv.org/abs/2402.05120 |
| Mixture-of-Agents | слоистая агрегация ответов | https://arxiv.org/abs/2406.04692 |
| Надёжность LLM-судьи | voting не лечит корреляцию; minority-veto | https://arxiv.org/abs/2412.12509 |
| PromptBoosting | AdaBoost на замороженных весах через промпты | https://arxiv.org/abs/2212.09257 |
| Language Models are Weak Learners | LLM как weak learner в boosting без дообучения | https://arxiv.org/abs/2306.14101 |

### Данные и метрики
| Работа | Что взяли | Ссылка |
|---|---|---|
| OmniSQL / SynSQL (back-translation) | синтез датасета SQL→Text | https://arxiv.org/abs/2503.02240 |
| ETM (Enhanced Tree Matching) | честная метрика эквивалентности SQL | https://arxiv.org/abs/2407.07313 |
| SecureSQL | метрики по типам атак | https://aclanthology.org/2024.findings-emnlp.346/ |
| Superviz25-SQL | шаблонный синтез + insider-кейсы | https://zenodo.org/records/17086037 |
| TKDE 2025 Survey | общая карта поля Text-to-SQL | https://arxiv.org/abs/2408.05109 |

> Итого задействовано **35+ источников**. В репозитории каждый разобран отдельной карточкой (`research/materials*/`), а решения зафиксированы в `docs/adr/` (ADR-0002…0012).
